# Rossmann Store Sales — Machine Learning Modeling

## Goal
Train regression models and compare them with the baseline from Notebook 04.

- Train on dates before July 2015 and validate on July 2015.
- Use open-store records only.
- Do not use `Customers`, because it is unknown when forecasting.
- Compare models with MAE and RMSE.


## 1. Setup and Data Loading
Load the cleaned sales and store data.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

In [2]:
candidate_roots = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "data" / "processed").exists()),
    Path.cwd().parent,
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DATA_DIR / "train_clean.parquet"
STORE_PATH = PROCESSED_DATA_DIR / "store_clean.parquet"

assert TRAIN_PATH.exists(), f"File not found: {TRAIN_PATH}"
assert STORE_PATH.exists(), f"File not found: {STORE_PATH}"

df = pd.read_parquet(TRAIN_PATH)
store_df = pd.read_parquet(STORE_PATH)

df["Date"] = pd.to_datetime(df["Date"])

print("Training data shape:", df.shape)
print("Store metadata shape:", store_df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Training data shape: (1017209, 9)
Store metadata shape: (1115, 10)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


## 2. Train-Validation Split
Use dates before July 2015 for training and July 2015 for validation.


In [3]:
feature_cols = ["Store", "DayOfWeek", "Promo"]
target_col = "Sales"
split_date = pd.Timestamp("2015-07-01")

train_data = df.loc[df["Date"] < split_date].copy()
valid_data = df.loc[df["Date"] >= split_date].copy()

train_open = train_data.loc[train_data["Open"] == 1].copy()
valid_open = valid_data.loc[valid_data["Open"] == 1].copy()

X_train = train_open[feature_cols].copy()
y_train = train_open[target_col].copy()

X_valid = valid_open[feature_cols].copy()
y_valid = valid_open[target_col].copy()

print("Training rows:", len(train_open))
print("Validation rows:", len(valid_open))
print("Last training date:", train_data["Date"].max())
print("First validation date:", valid_data["Date"].min())

assert train_data["Date"].max() < valid_data["Date"].min()
assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)

Training rows: 814204
Validation rows: 30188
Last training date: 2015-06-30 00:00:00
First validation date: 2015-07-01 00:00:00


## 3. Metrics and Baseline
Use open-day MAE and RMSE. Rebuild the `Store + DayOfWeek + Promo` baseline using training data only.


In [ ]:
def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
    }


def prepare_catboost_features(frame, features, categorical_columns):
    result = frame[features].copy()

    for column in categorical_columns:
        result[column] = result[column].astype(str)

    return result

In [5]:
baseline_group_columns = ["Store", "DayOfWeek", "Promo"]

baseline_group_means = (
    train_open.groupby(baseline_group_columns)["Sales"]
    .mean()
    .rename("PredictedSales")
    .reset_index()
)

store_mean_fallback = train_open.groupby("Store")["Sales"].mean()

baseline_predictions = valid_open[["Store", "DayOfWeek", "Promo", "Sales"]].merge(
    baseline_group_means,
    on=baseline_group_columns,
    how="left",
    validate="many_to_one",
)

baseline_fallback_count = baseline_predictions["PredictedSales"].isna().sum()

baseline_predictions["PredictedSales"] = baseline_predictions["PredictedSales"].fillna(
    baseline_predictions["Store"].map(store_mean_fallback)
)

if baseline_predictions["PredictedSales"].isna().any():
    raise ValueError("Some validation rows do not have a baseline prediction.")

baseline_metrics = regression_metrics(
    baseline_predictions["Sales"],
    baseline_predictions["PredictedSales"],
)

print("Baseline fallback rows:", baseline_fallback_count)
display(pd.DataFrame([baseline_metrics], index=["Historical Baseline"]).round(2))

Baseline fallback rows: 0


,MAE,RMSE
Historical Baseline,732.18,1038.49


The baseline predicts average sales for each store, weekday, and promotion group. Store average is used if a group is missing.


## 4. Linear Regression
One-hot encode `Store` and `DayOfWeek`; keep `Promo` as a numeric feature.


In [6]:
categorical_features = ["Store", "DayOfWeek"]

linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        ("binary", "passthrough", ["Promo"]),
    ],
)

linear_pipeline = Pipeline(
    steps=[
        ("preprocessing", linear_preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_pipeline.fit(X_train, y_train)

linear_predictions = linear_pipeline.predict(X_valid)
linear_metrics = regression_metrics(y_valid, linear_predictions)

display(pd.DataFrame([linear_metrics], index=["Linear Regression"]).round(2))

,MAE,RMSE
Linear Regression,911.27,1248.04


### Linear Regression Errors
Compare prediction errors on promotional and non-promotional days.


In [7]:
linear_error_analysis = valid_open[
    ["Store", "Date", "Promo", "DayOfWeek", "Sales"]
].copy()

linear_error_analysis["PredictedSales"] = linear_predictions
linear_error_analysis["SignedError"] = (
    linear_error_analysis["Sales"] - linear_error_analysis["PredictedSales"]
)
linear_error_analysis["AbsoluteError"] = linear_error_analysis["SignedError"].abs()

linear_error_by_promo = linear_error_analysis.groupby("Promo").agg(
    observation_count=("Sales", "size"),
    mae=("AbsoluteError", "mean"),
    mean_signed_error=("SignedError", "mean"),
)

display(linear_error_by_promo.round(2))

,observation_count,mae,mean_signed_error
Promo,,,
0,15717,926.89,53.35
1,14471,894.31,52.22


Linear Regression does not beat the historical baseline. The next model allows weekday patterns to differ by store.


## 5. Store-Weekday Interaction
Create a `StoreWeekday` feature to represent each store's weekday pattern.


In [8]:
X_train_interaction = X_train.copy()
X_valid_interaction = X_valid.copy()

for data in (X_train_interaction, X_valid_interaction):
    data["StoreWeekday"] = (
        data["Store"].astype(str) + "_" + data["DayOfWeek"].astype(str)
    )

interaction_preprocessor = ColumnTransformer(
    transformers=[
        (
            "store_weekday",
            OneHotEncoder(handle_unknown="ignore"),
            ["StoreWeekday"],
        ),
        ("promo", "passthrough", ["Promo"]),
    ],
)

interaction_pipeline = Pipeline(
    steps=[
        ("preprocessing", interaction_preprocessor),
        ("model", LinearRegression()),
    ]
)

interaction_pipeline.fit(
    X_train_interaction[["StoreWeekday", "Promo"]],
    y_train,
)

interaction_predictions = interaction_pipeline.predict(
    X_valid_interaction[["StoreWeekday", "Promo"]]
)

interaction_metrics = regression_metrics(y_valid, interaction_predictions)

display(
    pd.DataFrame(
        [linear_metrics, interaction_metrics],
        index=["Linear Regression", "Linear + StoreWeekday"],
    ).round(2)
)

,MAE,RMSE
Linear Regression,911.27,1248.04
Linear + StoreWeekday,809.48,1106.27


The interaction improves Linear Regression, but it still does not beat the baseline.


## 6. Decision Tree
Train one Decision Tree model using the initial features.


In [9]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessing", linear_preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=16,
                min_samples_leaf=50,
                random_state=42,
            ),
        ),
    ]
)

tree_pipeline.fit(X_train, y_train)

tree_predictions = tree_pipeline.predict(X_valid)
tree_metrics = regression_metrics(y_valid, tree_predictions)

display(pd.DataFrame([tree_metrics], index=["Decision Tree"]).round(2))

,MAE,RMSE
Decision Tree,1785.52,2346.5


The tested Decision Tree does not beat the baseline, so it is kept only as a comparison model.


## 7. CatBoost
Train CatBoost with `Store`, `DayOfWeek`, and `Promo`.


In [10]:
catboost_params = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.1,
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False,
}

catboost_categorical = ["Store", "DayOfWeek"]

X_train_cat = prepare_catboost_features(
    X_train,
    feature_cols,
    catboost_categorical,
)

X_valid_cat = prepare_catboost_features(
    X_valid,
    feature_cols,
    catboost_categorical,
)

catboost_model = CatBoostRegressor(
    **catboost_params,
    cat_features=catboost_categorical,
)

catboost_model.fit(X_train_cat, y_train)

catboost_predictions = catboost_model.predict(X_valid_cat)
catboost_metrics = regression_metrics(y_valid, catboost_predictions)

display(pd.DataFrame([catboost_metrics], index=["CatBoost"]).round(2))

,MAE,RMSE
CatBoost,867.76,1303.99


## 8. Calendar Features
Test `Month` and `SchoolHoliday` with the same CatBoost settings.


In [ ]:
X_train_month = X_train_cat.copy()
X_valid_month = X_valid_cat.copy()

X_train_month["Month"] = train_open["Date"].dt.month.astype(str)
X_valid_month["Month"] = valid_open["Date"].dt.month.astype(str)

month_categorical = catboost_categorical + ["Month"]

month_model = CatBoostRegressor(
    **catboost_params,
    cat_features=month_categorical,
)

month_model.fit(X_train_month, y_train)

month_predictions = month_model.predict(X_valid_month)
month_metrics = regression_metrics(y_valid, month_predictions)

In [12]:
X_train_school = X_train_cat.copy()
X_valid_school = X_valid_cat.copy()

X_train_school["SchoolHoliday"] = train_open["SchoolHoliday"].to_numpy()
X_valid_school["SchoolHoliday"] = valid_open["SchoolHoliday"].to_numpy()

school_model = CatBoostRegressor(
    **catboost_params,
    cat_features=catboost_categorical,
)

school_model.fit(X_train_school, y_train)

school_predictions = school_model.predict(X_valid_school)
school_metrics = regression_metrics(y_valid, school_predictions)

calendar_comparison = pd.DataFrame(
    [
        catboost_metrics,
        month_metrics,
        school_metrics,
    ],
    index=[
        "CatBoost",
        "CatBoost + Month",
        "CatBoost + SchoolHoliday",
    ],
)

display(calendar_comparison.round(2))

,MAE,RMSE
CatBoost,867.76,1303.99
CatBoost + Month,864.99,1296.51
CatBoost + SchoolHoliday,888.59,1313.19


Month gives only a small change, while SchoolHoliday worsens the July result. Neither version beats the baseline.


## 9. Store Metadata
Add `StoreType` and `Assortment` from `store.csv`.


In [13]:
metadata_columns = ["Store", "StoreType", "Assortment"]

all_open_enriched = df.loc[df["Open"] == 1].merge(
    store_df[metadata_columns],
    on="Store",
    how="left",
    validate="many_to_one",
)

assert len(all_open_enriched) == int((df["Open"] == 1).sum())
assert all_open_enriched[["StoreType", "Assortment"]].notna().all().all()

metadata_features = [
    "Store",
    "DayOfWeek",
    "Promo",
    "StoreType",
    "Assortment",
]

metadata_categorical = [
    "Store",
    "DayOfWeek",
    "StoreType",
    "Assortment",
]

metadata_train = all_open_enriched.loc[all_open_enriched["Date"] < split_date].copy()

metadata_valid = all_open_enriched.loc[all_open_enriched["Date"] >= split_date].copy()

X_train_metadata = prepare_catboost_features(
    metadata_train,
    metadata_features,
    metadata_categorical,
)

X_valid_metadata = prepare_catboost_features(
    metadata_valid,
    metadata_features,
    metadata_categorical,
)

print("Enriched training rows:", len(metadata_train))
print("Enriched validation rows:", len(metadata_valid))

Enriched training rows: 814204
Enriched validation rows: 30188


In [14]:
metadata_model = CatBoostRegressor(
    **catboost_params,
    cat_features=metadata_categorical,
)

metadata_model.fit(
    X_train_metadata,
    metadata_train["Sales"],
)

metadata_predictions = metadata_model.predict(X_valid_metadata)

metadata_metrics = regression_metrics(
    metadata_valid["Sales"],
    metadata_predictions,
)

display(pd.DataFrame([metadata_metrics], index=["CatBoost + Metadata"]).round(2))

,MAE,RMSE
CatBoost + Metadata,853.36,1260.13


Store metadata improves the initial CatBoost result, but the model remains behind the baseline.


## 10. Competition Features
Add competition distance and features based on the recorded competitor opening date. The year 1900 is treated as unknown for these derived features.


In [15]:
competition_columns = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
]

competition_missingness = store_df[competition_columns].isna().sum()
display(competition_missingness.to_frame("Missing_Values"))

print(
    "Stores with competition opening year 1900:",
    int((store_df["CompetitionOpenSinceYear"] == 1900).sum()),
)

,Missing_Values
CompetitionDistance,3
CompetitionOpenSinceMonth,354
CompetitionOpenSinceYear,354


Stores with competition opening year 1900: 1


In [ ]:
all_comp_enriched = all_open_enriched.merge(
    store_df[["Store", *competition_columns]],
    on="Store",
    how="left",
    validate="many_to_one",
)

comp_year = all_comp_enriched["CompetitionOpenSinceYear"].astype("float64")
comp_month = all_comp_enriched["CompetitionOpenSinceMonth"].astype("float64")

known_competition_date = comp_year.notna() & comp_month.notna() & (comp_year != 1900)

months_since_opening = (
    (all_comp_enriched["Date"].dt.year - comp_year) * 12
    + all_comp_enriched["Date"].dt.month
    - comp_month
)

all_comp_enriched["CompetitionDateKnown"] = known_competition_date.astype("int8")

all_comp_enriched["CompetitionActive"] = (
    (months_since_opening >= 0).astype("float64").where(known_competition_date)
)

all_comp_enriched["CompetitionAgeMonths"] = months_since_opening.clip(lower=0).where(
    known_competition_date
)

assert len(all_comp_enriched) == len(all_open_enriched)
assert (all_comp_enriched["CompetitionAgeMonths"].dropna() >= 0).all()

`CompetitionDistance` is the distance to the nearest competitor. `CompetitionActive` and `CompetitionAgeMonths` describe the recorded opening date. Historical availability of this metadata is not fully verified.


In [17]:
competition_features = metadata_features + [
    "CompetitionDistance",
    "CompetitionDateKnown",
    "CompetitionActive",
    "CompetitionAgeMonths",
]

competition_train = all_comp_enriched.loc[all_comp_enriched["Date"] < split_date].copy()

competition_valid = all_comp_enriched.loc[
    all_comp_enriched["Date"] >= split_date
].copy()

X_train_competition = prepare_catboost_features(
    competition_train,
    competition_features,
    metadata_categorical,
)

X_valid_competition = prepare_catboost_features(
    competition_valid,
    competition_features,
    metadata_categorical,
)

competition_model = CatBoostRegressor(
    **catboost_params,
    cat_features=metadata_categorical,
)

competition_model.fit(
    X_train_competition,
    competition_train["Sales"],
)

competition_predictions = competition_model.predict(X_valid_competition)

competition_metrics = regression_metrics(
    competition_valid["Sales"],
    competition_predictions,
)

display(
    pd.DataFrame(
        [
            metadata_metrics,
            competition_metrics,
        ],
        index=[
            "CatBoost + Metadata",
            "CatBoost + Competition",
        ],
    ).round(2)
)

,MAE,RMSE
CatBoost + Metadata,853.36,1260.13
CatBoost + Competition,790.69,1101.12


### Feature Importance Interpretation

The feature importance results show that the fitted CatBoost model relies most heavily on store identity and promotion status.

The top features are:

| Feature | Importance |
|---|---:|
| Store | 46.69 |
| Promo | 23.33 |
| CompetitionDistance | 11.03 |
| DayOfWeek | 8.91 |
| CompetitionAgeMonths | 4.57 |

`Store` and `Promo` have the highest importance scores, indicating that store-specific differences and promotion status provide substantial predictive information.

`CompetitionDistance` also contributes useful information to the model.

These importance scores describe the fitted CatBoost model with competition features. They do not represent causal effects and should not be interpreted as proof that a feature directly causes changes in sales.

Feature importance may also change when different features, models, or validation periods are used.

In [18]:
competition_importance = pd.Series(
    competition_model.get_feature_importance(),
    index=competition_features,
    name="Importance",
).sort_values(ascending=False)

display(competition_importance.round(2).to_frame())

,Importance
Store,46.69
Promo,23.33
CompetitionDistance,11.03
DayOfWeek,8.91
CompetitionAgeMonths,4.57
StoreType,2.55
Assortment,1.98
CompetitionActive,0.68
CompetitionDateKnown,0.25


### Monthly Check and Distance Ablation
Compare Metadata and Competition feature sets from April to July 2015. Also test the model without `CompetitionDistance`.


In [ ]:
def evaluate_catboost_folds(
    data,
    feature_sets,
    categorical_columns,
    validation_months,
    model_params,
):
    results = []

    for start_date in validation_months:
        end_date = start_date + pd.offsets.MonthBegin(1)

        fold_train = data.loc[data["Date"] < start_date]
        fold_valid = data.loc[(data["Date"] >= start_date) & (data["Date"] < end_date)]

        for model_name, features in feature_sets.items():
            X_fold_train = prepare_catboost_features(
                fold_train,
                features,
                categorical_columns,
            )

            X_fold_valid = prepare_catboost_features(
                fold_valid,
                features,
                categorical_columns,
            )

            model = CatBoostRegressor(
                **model_params,
                cat_features=categorical_columns,
            )

            model.fit(X_fold_train, fold_train["Sales"])
            predictions = model.predict(X_fold_valid)

            metrics = regression_metrics(
                fold_valid["Sales"],
                predictions,
            )

            results.append(
                {
                    "Month": start_date.strftime("%Y-%m"),
                    "Model": model_name,
                    **metrics,
                }
            )

    return pd.DataFrame(results)

In [20]:
validation_months = pd.to_datetime(
    [
        "2015-04-01",
        "2015-05-01",
        "2015-06-01",
        "2015-07-01",
    ]
)

features_without_distance = [
    column for column in competition_features if column != "CompetitionDistance"
]

competition_feature_sets = {
    "Metadata": metadata_features,
    "Competition": competition_features,
    "Competition without Distance": features_without_distance,
}

competition_cv = evaluate_catboost_folds(
    all_comp_enriched,
    competition_feature_sets,
    metadata_categorical,
    validation_months,
    catboost_params,
)

competition_cv_pivot = competition_cv.pivot(
    index="Month",
    columns="Model",
    values="MAE",
)

display(competition_cv_pivot.round(2))

Model,Competition,Competition without Distance,Metadata
Month,,,
2015-04,1056.01,1082.45,1111.12
2015-05,921.02,937.42,987.27
2015-06,844.42,867.13,905.54
2015-07,790.69,819.00,853.36


Competition features improve MAE over Metadata in all four months. Removing `CompetitionDistance` increases MAE in every month, so we keep it.


## 11. Promo2 Features
Create features for whether Promo2 has started and whether the current month is in its schedule.


In [ ]:
promo2_columns = [
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]

promo2_store = store_df[["Store", *promo2_columns]].copy()

is_promo2_participant = promo2_store["Promo2"] == 1

iso_week = (
    promo2_store.loc[is_promo2_participant, "Promo2SinceYear"]
    .astype("int64")
    .astype(str)
    + "-W"
    + promo2_store.loc[is_promo2_participant, "Promo2SinceWeek"]
    .astype("int64")
    .astype(str)
    .str.zfill(2)
    + "-1"
)

promo2_store["Promo2StartDate"] = pd.Series(
    pd.NaT,
    index=promo2_store.index,
    dtype="datetime64[ns]",
)

promo2_store.loc[is_promo2_participant, "Promo2StartDate"] = pd.to_datetime(
    iso_week,
    format="%G-W%V-%u",
    errors="raise",
)

assert (
    promo2_store.loc[
        is_promo2_participant,
        "Promo2StartDate",
    ]
    .notna()
    .all()
)

In [ ]:
all_promo2_enriched = all_comp_enriched.merge(
    promo2_store[["Store", "Promo2", "PromoInterval", "Promo2StartDate"]],
    on="Store",
    how="left",
    validate="many_to_one",
)

promo2_started = (all_promo2_enriched["Promo2"] == 1) & (
    all_promo2_enriched["Date"] >= all_promo2_enriched["Promo2StartDate"]
)

promo_months = {
    "Jan,Apr,Jul,Oct": [1, 4, 7, 10],
    "Feb,May,Aug,Nov": [2, 5, 8, 11],
    "Mar,Jun,Sept,Dec": [3, 6, 9, 12],
}

observation_month = all_promo2_enriched["Date"].dt.month
promo_interval = all_promo2_enriched["PromoInterval"]

scheduled_month = pd.Series(
    False,
    index=all_promo2_enriched.index,
)

for interval_name, months in promo_months.items():
    scheduled_month |= promo_interval.eq(interval_name) & observation_month.isin(months)

all_promo2_enriched["Promo2Started"] = promo2_started.astype("int8")

all_promo2_enriched["Promo2Scheduled"] = (promo2_started & scheduled_month).astype(
    "int8"
)

assert (
    all_promo2_enriched["Promo2Scheduled"] <= all_promo2_enriched["Promo2Started"]
).all()

assert (
    (
        all_promo2_enriched.loc[
            all_promo2_enriched["Promo2"] == 0,
            ["Promo2Started", "Promo2Scheduled"],
        ]
        == 0
    )
    .all()
    .all()
)

`Promo2Started` checks the recorded start date. `Promo2Scheduled` also checks the store's listed promotional months.


In [23]:
promo2_features = competition_features + [
    "Promo2Started",
    "Promo2Scheduled",
]

promo2_train = all_promo2_enriched.loc[all_promo2_enriched["Date"] < split_date].copy()

promo2_valid = all_promo2_enriched.loc[all_promo2_enriched["Date"] >= split_date].copy()

X_train_promo2 = prepare_catboost_features(
    promo2_train,
    promo2_features,
    metadata_categorical,
)

X_valid_promo2 = prepare_catboost_features(
    promo2_valid,
    promo2_features,
    metadata_categorical,
)

promo2_model = CatBoostRegressor(
    **catboost_params,
    cat_features=metadata_categorical,
)

promo2_model.fit(X_train_promo2, promo2_train["Sales"])

promo2_predictions = promo2_model.predict(X_valid_promo2)

promo2_metrics = regression_metrics(
    promo2_valid["Sales"],
    promo2_predictions,
)

display(
    pd.DataFrame(
        [
            competition_metrics,
            promo2_metrics,
        ],
        index=[
            "CatBoost + Competition",
            "CatBoost + Competition + Promo2",
        ],
    ).round(2)
)

,MAE,RMSE
CatBoost + Competition,790.69,1101.12
CatBoost + Competition + Promo2,794.46,1108.17


Promo2 slightly worsens the July result, so these features are not used in the selected model.


## 12. Historical Sales Features
Add three features based only on sales before each forecast month:
- `Sales_Mean_30D`
- `Sales_Mean_90D`
- `Sales_Mean_Store_DOW`

The features are fixed for the full forecast month to avoid using sales from that month.


In [ ]:
def add_monthly_historical_sales_features(target, history):
    target_features = target.copy()
    target_features["Date"] = pd.to_datetime(target_features["Date"])
    target_features["_row_position"] = np.arange(len(target_features))
    target_features["_forecast_month"] = (
        target_features["Date"].dt.to_period("M").dt.to_timestamp()
    )

    history_open = history.loc[
        history["Open"] == 1,
        ["Store", "Date", "DayOfWeek", "Sales"],
    ].copy()
    history_open["Date"] = pd.to_datetime(history_open["Date"])

    feature_parts = []

    for forecast_month, month_target in target_features.groupby(
        "_forecast_month",
        sort=True,
    ):
        month_target = month_target.copy()
        available_history = history_open.loc[history_open["Date"] < forecast_month]

        recent_30_days = available_history.loc[
            available_history["Date"] >= forecast_month - pd.Timedelta(days=30)
        ]

        recent_90_days = available_history.loc[
            available_history["Date"] >= forecast_month - pd.Timedelta(days=90)
        ]

        mean_30_days = recent_30_days.groupby("Store")["Sales"].mean()
        mean_90_days = recent_90_days.groupby("Store")["Sales"].mean()

        mean_store_weekday = available_history.groupby(["Store", "DayOfWeek"])[
            "Sales"
        ].mean()

        month_target["Sales_Mean_30D"] = month_target["Store"].map(mean_30_days)
        month_target["Sales_Mean_90D"] = month_target["Store"].map(mean_90_days)

        target_keys = pd.MultiIndex.from_frame(month_target[["Store", "DayOfWeek"]])

        month_target["Sales_Mean_Store_DOW"] = mean_store_weekday.reindex(
            target_keys
        ).to_numpy()

        feature_parts.append(month_target)

    result = (
        pd.concat(feature_parts)
        .sort_values("_row_position")
        .drop(columns=["_row_position", "_forecast_month"])
    )

    return result

In [25]:
historical_feature_columns = [
    "Sales_Mean_30D",
    "Sales_Mean_90D",
    "Sales_Mean_Store_DOW",
]

train_with_history = add_monthly_historical_sales_features(
    competition_train,
    competition_train,
)

valid_with_history = add_monthly_historical_sales_features(
    competition_valid,
    competition_train,
)

assert len(train_with_history) == len(competition_train)
assert len(valid_with_history) == len(competition_valid)

X_train_competition_history = X_train_competition.copy()
X_valid_competition_history = X_valid_competition.copy()

X_train_competition_history[historical_feature_columns] = train_with_history[
    historical_feature_columns
].to_numpy()

X_valid_competition_history[historical_feature_columns] = valid_with_history[
    historical_feature_columns
].to_numpy()

historical_feature_missingness = pd.DataFrame(
    {
        "Training Missing": (
            train_with_history[historical_feature_columns].isna().sum()
        ),
        "Validation Missing": (
            valid_with_history[historical_feature_columns].isna().sum()
        ),
    }
)

display(historical_feature_missingness)

,Training Missing,Validation Missing
Sales_Mean_30D,33490,0
Sales_Mean_90D,33441,0
Sales_Mean_Store_DOW,29001,0


### Historical Feature Availability

Some training observations have missing values in the historical sales features.

| Feature | Training Missing | Validation Missing |
|---|---:|---:|
| Sales_Mean_30D | 33,490 | 0 |
| Sales_Mean_90D | 33,441 | 0 |
| Sales_Mean_Store_DOW | 29,001 | 0 |

These missing values occur when sufficient historical observations are not available before the corresponding forecast month.

For example, a store may not have enough previous observations to calculate a 30-day or 90-day historical average.

These values are not filled using future sales. Instead, they are preserved as missing numerical features.

CatBoost supports missing numerical feature values, so no additional imputation is applied in this experiment.

The validation dataset has no missing values in these three historical features.

In [26]:
historical_model = CatBoostRegressor(
    **catboost_params,
    cat_features=metadata_categorical,
)

historical_model.fit(
    X_train_competition_history,
    competition_train["Sales"],
)

historical_predictions = historical_model.predict(X_valid_competition_history)

historical_metrics = regression_metrics(
    competition_valid["Sales"],
    historical_predictions,
)

historical_experiment_comparison = pd.DataFrame(
    [
        baseline_metrics,
        competition_metrics,
        historical_metrics,
    ],
    index=[
        "Historical Baseline",
        "CatBoost + Competition",
        "CatBoost + Competition + Historical Sales",
    ],
)

display(historical_experiment_comparison.round(2))

print(
    "MAE improvement vs CatBoost + Competition:",
    round(competition_metrics["MAE"] - historical_metrics["MAE"], 2),
)

print(
    "MAE improvement vs Historical Baseline:",
    round(baseline_metrics["MAE"] - historical_metrics["MAE"], 2),
)

,MAE,RMSE
Historical Baseline,732.18,1038.49
CatBoost + Competition,790.69,1101.12
CatBoost + Competition + Historical Sales,718.09,991.79


MAE improvement vs CatBoost + Competition: 72.6
MAE improvement vs Historical Baseline: 14.09


The selected CatBoost model with Competition and Historical Sales has July MAE **718.09**. The baseline MAE is **732.18**, so the model improves MAE by about **1.92%**. Notebook 06 checks whether this result holds across more months.


## 13. Final Model Comparison
Compare all tested models on the same July 2015 open-store validation data.


In [27]:
model_results = {
    "Historical Baseline": baseline_metrics,
    "Linear Regression": linear_metrics,
    "Linear + StoreWeekday": interaction_metrics,
    "Decision Tree": tree_metrics,
    "CatBoost": catboost_metrics,
    "CatBoost + Month": month_metrics,
    "CatBoost + SchoolHoliday": school_metrics,
    "CatBoost + Metadata": metadata_metrics,
    "CatBoost + Competition": competition_metrics,
    "CatBoost + Competition + Promo2": promo2_metrics,
    "CatBoost + Competition + Historical Sales": historical_metrics,
}

final_comparison = pd.DataFrame(model_results).T.rename_axis("Model").sort_values("MAE")

final_comparison["MAE_vs_Baseline_Pct"] = (
    (final_comparison["MAE"] - baseline_metrics["MAE"]) / baseline_metrics["MAE"] * 100
)

display(final_comparison.round(2))

,MAE,RMSE,MAE_vs_Baseline_Pct
Model,,,
CatBoost + Competition + Historical Sales,718.09,991.79,-1.92
Historical Baseline,732.18,1038.49,0.00
CatBoost + Competition,790.69,1101.12,7.99
CatBoost + Competition + Promo2,794.46,1108.17,8.51
Linear + StoreWeekday,809.48,1106.27,10.56
CatBoost + Metadata,853.36,1260.13,16.55
CatBoost + Month,864.99,1296.51,18.14
CatBoost,867.76,1303.99,18.52
CatBoost + SchoolHoliday,888.59,1313.19,21.36


In [28]:
machine_learning_results = final_comparison.drop(index="Historical Baseline")

best_ml_name = machine_learning_results["MAE"].idxmin()
best_ml_mae = machine_learning_results.loc[best_ml_name, "MAE"]

print("Best machine learning model:", best_ml_name)
print("Best machine learning MAE:", round(best_ml_mae, 2))
print("Historical baseline MAE:", round(baseline_metrics["MAE"], 2))

if best_ml_mae < baseline_metrics["MAE"]:
    print("The best machine learning model outperformed the historical baseline.")
else:
    print("The historical baseline remains stronger on this validation period.")

Best machine learning model: CatBoost + Competition + Historical Sales
Best machine learning MAE: 718.09
Historical baseline MAE: 732.18
The best machine learning model outperformed the historical baseline.


## 14. Conclusion

Several machine learning approaches were evaluated against the historical baseline using July 2015 open-store validation data.

The best-performing machine learning model is:

**CatBoost + Competition + Historical Sales**

### Final Results

| Metric | Historical Baseline | Selected CatBoost |
|---|---:|---:|
| MAE | 732.18 | 718.09 |
| RMSE | 1,038.49 | 991.79 |

The selected CatBoost model reduces MAE by approximately 1.92% compared with the historical baseline.

In absolute terms, the MAE decreases by 14.09 sales units.

Adding historical sales features improves the Competition-based CatBoost model's MAE from 790.69 to 718.09.

### Important Findings

- Store-specific information is important for predicting sales.
- Competition features improve performance compared with the initial store metadata model.
- Adding Promo2 features slightly worsens the July validation result.
- Historical sales features provide the largest improvement in the final modeling experiments.
- The selected model is the only tested machine learning configuration that outperforms the historical baseline on July 2015.

### Limitations

The improvement over the baseline is relatively small and is measured on a single validation month.

The historical availability of competition metadata has not been fully verified.

In addition, multiple feature and model experiments were conducted using development validation data. Therefore, these results should not be interpreted as an unbiased final estimate of future performance.

Notebook 06 evaluates the selected model across April–July 2015 to check whether the improvement is consistent across multiple months.

The current project reports MAE and RMSE. These metrics are useful for model comparison, but they are not the official evaluation metric of the original Rossmann Kaggle competition.